# 25 — ABSA database check

Inspect everything the ABSA pipeline has written to `data/hotel_reviews.db`:
labeling progress, aspect/sentiment distributions, evidence quality, coined
sub_aspects, roll-ups, gold tables, and a browsable sample of labeled reviews.

All cells are **read-only**.

> ⚠️ DuckDB is single-writer: even a read-only kernel connection blocks
> `absa_label_retrieve.py` from taking the write lock. **Close/restart this
> kernel (or run the last cell) before retrieving a batch.**

In [1]:
import sys
sys.path.insert(0, "../src")

import duckdb
import pandas as pd

from absa_label import DB_PATH

pd.set_option("display.max_colwidth", 120)
con = duckdb.connect(str(DB_PATH), read_only=True)

con.execute("""
    SELECT table_name, table_type FROM information_schema.tables
    WHERE table_name LIKE 'ABSA%' OR table_name LIKE 'REVIEW_ASPECT%'
       OR table_name LIKE 'GOLD%' OR table_name = 'SENTENCE_LABELS'
    ORDER BY 1
""").df()

,table_name,table_type
0,ABSA_SAMPLE,BASE TABLE
1,GOLD_REVIEW_ASPECTS,BASE TABLE
2,GOLD_TRIPADVISOR,BASE TABLE
3,REVIEW_ASPECTS,BASE TABLE
4,REVIEW_ASPECT_ROLLUP,BASE TABLE


## 1. Labeling progress — how far through the 15k sample?

In [2]:
con.execute("""
    SELECT s.split, s.language, s.source,
           count(*)                          AS sampled,
           count(a.review_id)                AS labeled,
           round(count(a.review_id) * 100.0 / count(*), 1) AS pct
    FROM ABSA_SAMPLE s
    LEFT JOIN (SELECT DISTINCT review_id FROM REVIEW_ASPECTS) a USING (review_id)
    GROUP BY 1, 2, 3 ORDER BY 1, 2, 3
""").df()

,split,language,source,sampled,labeled,pct
0,test,en,agoda,286,172,60.1
1,test,en,googlemaps,234,46,19.7
2,test,vi,agoda,199,129,64.8
3,test,vi,googlemaps,793,171,21.6
4,train,en,agoda,2254,1416,62.8
5,train,en,googlemaps,1954,343,17.6
6,train,vi,agoda,1506,958,63.6
7,train,vi,googlemaps,6259,1233,19.7
8,val,en,agoda,269,180,66.9
9,val,en,googlemaps,263,36,13.7


In [3]:
# Overall + what the next wave will pick up
con.execute("""
    SELECT count(*) AS sampled,
           count(a.review_id) AS labeled,
           count(*) - count(a.review_id) AS remaining_for_next_waves
    FROM ABSA_SAMPLE s
    LEFT JOIN (SELECT DISTINCT review_id FROM REVIEW_ASPECTS) a USING (review_id)
""").df()

,sampled,labeled,remaining_for_next_waves
0,15000,4944,10056


## 2. Aspect × sentiment matrix (the core result)

In [4]:
con.execute("""
    PIVOT (SELECT key_aspect, sentiment FROM REVIEW_ASPECTS)
    ON sentiment IN ('positive', 'neutral', 'negative')
    USING count(*) GROUP BY key_aspect ORDER BY key_aspect
""").df()

,key_aspect,positive,neutral,negative
0,amenity,2254,93,777
1,experience,2875,164,876
2,facility,2213,151,2200
3,loyalty,800,2,208
4,other,15,2,11
5,service,3273,140,1272


In [5]:
# Evidence quality: substring-validity rate per aspect (hallucination check)
con.execute("""
    SELECT key_aspect, count(*) AS rows,
           sum(CASE WHEN NOT evidence_valid THEN 1 ELSE 0 END) AS invalid,
           round(sum(CASE WHEN NOT evidence_valid THEN 1 ELSE 0 END) * 100.0
                 / count(*), 2) AS invalid_pct
    FROM REVIEW_ASPECTS GROUP BY 1 ORDER BY 2 DESC
""").df()

,key_aspect,rows,invalid,invalid_pct
0,service,4685,20.0,0.43
1,facility,4564,22.0,0.48
2,experience,3915,15.0,0.38
3,amenity,3124,14.0,0.45
4,loyalty,1010,4.0,0.40
5,other,28,1.0,3.57


## 3. Sub-aspect vocabulary — seed usage + coined strings

In [6]:
# Top sub_aspects overall
con.execute("""
    SELECT key_aspect, sub_aspect, count(*) AS n
    FROM REVIEW_ASPECTS GROUP BY 1, 2 ORDER BY n DESC LIMIT 20
""").df()

,key_aspect,sub_aspect,n
0,service,staff_attitude,2062
1,amenity,local_convenience,1332
2,facility,facility_cleanliness,1202
3,experience,overall_satisfaction,1185
4,facility,room_features,1081
5,service,food_breakfast,1075
6,experience,price_value,920
7,facility,technical_equipment,774
8,amenity,leisure_facilities,571
9,facility,facility_condition,558


In [7]:
# Every coined sub_aspect (key_aspect='other') with an example evidence -
# candidates for promoting into the canonical vocabulary later
con.execute("""
    SELECT sub_aspect, count(*) AS n,
           any_value(sentiment) AS example_sentiment,
           any_value(evidence)  AS example_evidence
    FROM REVIEW_ASPECTS WHERE key_aspect = 'other'
    GROUP BY 1 ORDER BY n DESC
""").df()

,sub_aspect,n,example_sentiment,example_evidence
0,food_quality,5,positive,Dùng buổi trưa tại đây rất ngon
1,location,3,neutral,the location was decent
2,restaurant_food,2,positive,rất thích đồ ăn của nhà hàng
3,food_policy,1,negative,prohibits all food taken from outsise such as instant noodles & cakes
4,local_hospitality,1,negative,vietnam people are not really tourists friendly
5,location_quality,1,positive,Vụ trí cho 5 sao
6,luggage_storage,1,positive,storage room for your bags
7,conference_facilities,1,positive,Hội trường được ngăn bởi các vách ngăn di động cho phù hợp với các buổi tiệc có số lượng người ít hoặc nhiều
8,pest_issue,1,negative,Mùa mưa hok tránh khỏi muỗi
9,smoking_policy,1,neutral,đây là khách sạn không hút thuốc trong phòng


## 4. Browse labeled reviews — full text + every extracted aspect

In [8]:
N_BROWSE = 5  # rerun the cell for a new random draw

sample_ids = [r[0] for r in con.execute("""
    SELECT DISTINCT review_id FROM REVIEW_ASPECTS ORDER BY random() LIMIT ?
""", [N_BROWSE]).fetchall()]

for rid in sample_ids:
    hotel, lang, text = con.execute(
        "SELECT hotel_name, language, review_text FROM REVIEW_DATA WHERE review_id = ?",
        [rid]).fetchone()
    print(f"=== [{lang}] {hotel}  ({rid}) ===")
    print(" ", " ".join(text.split())[:300])
    for k, sub, sen, ev, ok in con.execute("""
        SELECT key_aspect, sub_aspect, sentiment, evidence, evidence_valid
        FROM REVIEW_ASPECTS WHERE review_id = ? ORDER BY aspect_rank
    """, [rid]).fetchall():
        flag = "" if ok else "  << INVALID EVIDENCE"
        print(f"    {k:11} <- {sub:26} {sen:8} | {ev[:70]!r}{flag}")
    print()

=== [vi] Asia Paradise Hotel Nha Trang  (ChZDSUhNMG9nS0VJQ0FnSURON0kzV0N3EAE) ===
  Tôi đặt phòng qua một công ty du lịch và ở lại hai đêm. Khách sạn nằm ở khu vực sầm uất của thành phố, có rất nhiều nhà hàng và cửa hàng tiện lợi gần đó, nên bạn sẽ không gặp khó khăn gì trong việc tìm đồ ăn hoặc mua sắm. Trung tâm du lịch …
    amenity     <- local_convenience          positive | 'có rất nhiều nhà hàng và cửa hàng tiện lợi gần đó'

=== [vi] Yasaka Saigon Resort Hotel & Spa  (99ff04cf-2f27-4706-b231-e07b0a584509) ===
  Tòa nhà tính phí gửi xe 100k/24 tiếng. Chi phí không cao nhưng rất bực mình khi không gộp chi phí này vào giá phòng.
    amenity     <- parking_facility           negative | 'Tòa nhà tính phí gửi xe 100k/24 tiếng'
    amenity     <- payment_billing            negative | 'rất bực mình khi không gộp chi phí này vào giá phòng'

=== [vi] River Hotel Ha Tien  (ChZDSUhNMG9nS0VJQ0FnSUMyNnRqbVNBEAE) ===
  Mùa dịch làm khách sạn tơi tả quá, mà được cái view ko chê vào đâu được, vẫ

## 5. Roll-up sanity — per-review macro sentiment vs overall rating

In [9]:
con.execute("SELECT * FROM REVIEW_ASPECT_ROLLUP LIMIT 10").df()

,review_id,asp5_facility,asp5_amenity,asp5_service,asp5_experience,asp5_loyalty,n_aspects,label_model,labeled_at
0,1c845bbb-5204-4e38-805b-b716c6e6bba8,NaN,negative,positive,NaN,None,2,claude-sonnet-5,2026-07-17 14:21:33.586496
1,1c8c73d6-f5ac-4275-bc5f-dca62daff2ec,NaN,NaN,positive,positive,None,2,claude-sonnet-5,2026-07-17 14:21:33.591246
2,1caaf908-44cf-428b-af90-3b127cec03d9,positive,negative,negative,positive,None,6,claude-sonnet-5,2026-07-17 14:21:33.602939
3,1cb26f4b-6773-4e38-8638-1ae43d8862e6,positive,negative,positive,positive,None,5,claude-sonnet-5,2026-07-17 14:21:33.612367
4,1cd449ba-35d1-47ba-a509-e0ddd5939179,positive,positive,positive,NaN,None,3,claude-sonnet-5,2026-07-17 14:21:33.619415
5,1cedb4bd-9663-438d-b9ec-20a39628de11,positive,positive,NaN,NaN,None,2,claude-sonnet-5,2026-07-17 14:21:33.623883
6,1cf640ef-4bff-43f4-80f0-3fe1a0db63ad,negative,positive,negative,NaN,None,3,claude-sonnet-5,2026-07-17 14:21:33.630455
7,1cf8addd-094f-40b0-a92b-99b7e3187ebd,positive,negative,NaN,positive,None,3,claude-sonnet-5,2026-07-17 14:21:33.636851
8,1d0e681d-e5fa-419f-a57b-6624ab96bf7e,positive,positive,NaN,positive,None,6,claude-sonnet-5,2026-07-17 14:21:33.648729
9,1d1928bc-206f-437b-9005-e9195cbb6959,NaN,negative,NaN,NaN,None,1,claude-sonnet-5,2026-07-17 14:21:33.652125


In [10]:
# Weak sanity signal: reviews the model calls facility-negative should have a
# lower overall reviewer rating than facility-positive ones
con.execute("""
    SELECT r.asp5_facility AS facility_sentiment,
           count(*) AS n,
           round(avg(d.rating_normalized), 2) AS avg_reviewer_rating_of5
    FROM REVIEW_ASPECT_ROLLUP r JOIN REVIEW_DATA d USING (review_id)
    WHERE r.asp5_facility IS NOT NULL AND d.rating_normalized IS NOT NULL
    GROUP BY 1 ORDER BY 3 DESC
""").df()

,facility_sentiment,n,avg_reviewer_rating_of5
0,positive,1316,4.57
1,neutral,326,4.10
2,negative,1094,3.04


## 6. Silver vs gold — quick agreement preview (GMap star tags)

For reviews that have BOTH a silver roll-up and a reviewer star tag, how often
do they agree? (Full evaluation comes later; this is the early temperature check.)

In [11]:
con.execute("""
    WITH pairs AS (
        SELECT g.review_id, g.key_aspect, g.sentiment AS gold,
               CASE g.key_aspect
                    WHEN 'facility' THEN r.asp5_facility
                    WHEN 'service'  THEN r.asp5_service
                    WHEN 'amenity'  THEN r.asp5_amenity END AS silver
        FROM GOLD_REVIEW_ASPECTS g
        JOIN REVIEW_ASPECT_ROLLUP r USING (review_id)
        WHERE g.gold_source = 'gmap_tag'
    )
    SELECT key_aspect,
           count(*) FILTER (WHERE silver IS NOT NULL)        AS both_present,
           round(avg(CASE WHEN silver = gold THEN 1.0 ELSE 0.0 END)
                 FILTER (WHERE silver IS NOT NULL) * 100, 1) AS agreement_pct
    FROM pairs GROUP BY 1 ORDER BY 1
""").df()

,key_aspect,both_present,agreement_pct
0,amenity,192,75.5
1,facility,213,59.6
2,service,262,85.1


## 7. Gold tables + sentence bridge status

In [12]:
# Gold + bridge tables (tolerant of tables not built yet)
existing = {r[0] for r in con.execute("""
    SELECT table_name FROM information_schema.tables
""").fetchall()}

parts = []
if "GOLD_REVIEW_ASPECTS" in existing:
    parts.append("SELECT 'GOLD_REVIEW_ASPECTS' AS tbl, gold_source AS detail, count(*) AS rows FROM GOLD_REVIEW_ASPECTS GROUP BY 2")
if "GOLD_TRIPADVISOR" in existing:
    parts.append("SELECT 'GOLD_TRIPADVISOR', 'spans', count(*) FROM GOLD_TRIPADVISOR")
if "SENTENCE_LABELS" in existing:
    parts.append("SELECT 'SENTENCE_LABELS', 'rows (built by absa_bridge.py)', count(*) FROM SENTENCE_LABELS")
else:
    print("SENTENCE_LABELS not built yet - run src/absa_bridge.py after labeling")

con.execute(" UNION ALL ".join(parts) + " ORDER BY 1, 2").df()


SENTENCE_LABELS not built yet - run src/absa_bridge.py after labeling


,tbl,detail,rows
0,GOLD_REVIEW_ASPECTS,gmap_highlight,31634
1,GOLD_REVIEW_ASPECTS,gmap_tag,94968
2,GOLD_TRIPADVISOR,spans,54326


In [13]:
con.close()
print("connection closed - safe to run retrieve now")

connection closed - safe to run retrieve now
